# DataStore smoke test

Uses `DataStore` from `DataStore.py` (camelCase API). The SQLite file `example_market.db` is created next to this notebook.

In [1]:
from pathlib import Path
import sys
from datetime import datetime, timezone

# Resolve folder containing DataStore.py (cwd when notebook lives there, else explicit fallback)
ROOT = Path.cwd()
if not (ROOT / "DataStore.py").exists():
    ROOT = Path(r"C:\Users\A715-72G\Documents\workspace\PythonInvestment\DatasetEngine")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from DataStore import DataStore

DB_FILE = ROOT / "example_market.db"
print("ROOT:", ROOT)
print("DB_FILE:", DB_FILE)

ROOT: c:\Users\A715-72G\Documents\workspace\PythonInvestment\DatasetEngine
DB_FILE: c:\Users\A715-72G\Documents\workspace\PythonInvestment\DatasetEngine\example_market.db


In [2]:
# Init schema, upsert one bar, insert one trade, read back
ds = DataStore(DB_FILE, initSchema=True)

ts = datetime.now(timezone.utc).isoformat()
ds.datasetUpsertBar("AAPL", ts, 185.0, 185.2, 175.5, 185.8, 180.6, 1_255_000.0)
ds.commit()

tid = ds.tradelogCreate(
    product="AAPL",
    side="long",
    entry_timestamp=ts,
    exit_timestamp=ts,
    entry_price=185.0,
    exit_price=151.0,
    result="TP",
    tp_pips=20.0,
    sl_pips=10.0,
    real_tp_pips=20.0,
    entry_strategy="ma_cross",
    timeframe="15m",
    entry_to_exit_minutes=45.0,
    percent_risk=1.0,
    pnl_percent=2.5,
    notes="notebook demo",
)
ds.commit()

print("bars:", ds.datasetGetBars("AAPL"))
print("trade id:", tid)
print("trade:", ds.getTradeLog(tid))
ds.close()

bars: [{'symbol': 'AAPL', 'timestamp': '2026-05-04T13:18:38.196234+00:00', 'open': 180.0, 'high': 181.2, 'low': 179.5, 'close': 180.8, 'adjClose': 180.6, 'volume': 1250000.0}, {'symbol': 'AAPL', 'timestamp': '2026-05-04T13:19:13.151245+00:00', 'open': 180.0, 'high': 181.2, 'low': 179.5, 'close': 180.8, 'adjClose': 180.6, 'volume': 1250000.0}, {'symbol': 'AAPL', 'timestamp': '2026-05-04T13:22:26.305065+00:00', 'open': 185.0, 'high': 185.2, 'low': 175.5, 'close': 185.8, 'adjClose': 180.6, 'volume': 1255000.0}, {'symbol': 'AAPL', 'timestamp': '2026-05-04T13:22:53.773153+00:00', 'open': 185.0, 'high': 185.2, 'low': 175.5, 'close': 185.8, 'adjClose': 180.6, 'volume': 1255000.0}]
trade id: 4
trade: {'id': 4, 'product': 'AAPL', 'side': 'long', 'entry_timestamp': '2026-05-04T13:22:53.773153+00:00', 'exit_timestamp': '2026-05-04T13:22:53.773153+00:00', 'entry_price': 185.0, 'exit_price': 151.0, 'result': 'TP', 'tp_pips': 20.0, 'sl_pips': 10.0, 'rr': 2.0, 'real_tp_pips': 20.0, 'actual_rr': 2.0, 

In [3]:
# Context manager + injected SQL (parameterized)
with DataStore(DB_FILE, initSchema=False) as ds:
    rows = ds.fetchAll(
        "SELECT symbol, COUNT(*) AS n FROM Dataset GROUP BY symbol"
    )
    print("counts:", rows)
    one = ds.fetchOne("SELECT * FROM TradeLog ORDER BY id DESC LIMIT 1", ())
    print("last trade:", one)

counts: [{'symbol': 'AAPL', 'n': 4}]
last trade: {'id': 4, 'product': 'AAPL', 'side': 'long', 'entry_timestamp': '2026-05-04T13:22:53.773153+00:00', 'exit_timestamp': '2026-05-04T13:22:53.773153+00:00', 'entry_price': 185.0, 'exit_price': 151.0, 'result': 'TP', 'tp_pips': 20.0, 'sl_pips': 10.0, 'rr': 2.0, 'real_tp_pips': 20.0, 'actual_rr': 2.0, 'entry_strategy': 'ma_cross', 'timeframe': '15m', 'entry_to_exit_minutes': 45.0, 'percent_risk': 1.0, 'pnl_percent': 2.5, 'notes': 'notebook demo', 'created_at': '2026-05-04 13:22:53', 'updated_at': '2026-05-04 13:22:53'}


In [ ]:
# Optional: remove test database (uncomment to run)
# DB_FILE.unlink(missing_ok=True)
# print("removed", DB_FILE)